# Chapter 1 — Setup & Environment

**Goal:** Get every dependency installed, download our base model, and verify DeepSeek API access.

By the end of this chapter you'll have:
- A working Python environment with all ML libraries
- A small base model loaded and ready for fine-tuning (SmolLM 135M)
- DeepSeek API configured (our teacher model for later chapters)

## 1.1 Install Dependencies

Run this once. On Colab, uncomment the first line.

In [ ]:
# !pip install torch transformers datasets peft accelerate bitsandbytes openai sentencepiece tokenizers huggingface_hub

## 1.2 Verify GPU / Device

We'll use GPU if available, CPU otherwise. Fine-tuning on CPU is slow but works for tiny models.

In [ ]:
import torch

if torch.cuda.is_available():
    device = "cuda"
    gpu_name = torch.cuda.get_device_name(0)
    print(f"GPU: {gpu_name}")
else:
    device = "cpu"
    print("No GPU found — using CPU. Fine-tuning will be slow but works.")

print(f"Using device: {device}")

## 1.3 Choose & Download the Base Model

We're using **SmolLM 135M** — a truly tiny model (~135M parameters, ~270MB). It's small enough to:
- Fine-tune on a single GPU in minutes
- Run on a laptop CPU
- Eventually quantize and deploy to mobile

Alternatives if you want to swap later: `google/mobilebert-uncased` (even smaller, ~25M) or `TinyLlama/TinyLlama-1.1B-Chat-v1.0` (larger, more capable).

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

BASE_MODEL = "HuggingFaceTB/SmolLM-135M"

print(f"Loading tokenizer for {BASE_MODEL}...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

# SmolLM's tokenizer may not have a pad token by default
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"Loading model ({'GPU' if device == 'cuda' else 'CPU'})...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None,
)

if device == "cpu":
    model = model.to(device)

params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Model loaded. Parameters: {params:.1f}M")

## 1.4 Quick Inference Test

Let's see what the untrained base model outputs. It should produce plausible but unfocused text — this is our starting point.

In [ ]:
def generate(prompt, max_tokens=50):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print(generate("The capital of France is"))
print("---")
print(generate("Once upon a time,"))

## 1.5 Configure DeepSeek API

DeepSeek will be our **teacher model** — we'll use it in Chapter 5 to generate high-quality training data. For now, we just verify the connection works.

Set your API key as an environment variable. **Never hardcode keys in notebooks or commit them.**

In [ ]:
import os
from openai import OpenAI

# Set your key before running. On Colab, use the "Secrets" panel (left sidebar, key icon).
# os.environ["DEEPSEEK_API_KEY"] = "sk-your-key-here"

api_key = os.environ.get("DEEPSEEK_API_KEY")
if not api_key:
    print("DEEPSEEK_API_KEY not set. Set it to test the teacher model connection.")
else:
    client = OpenAI(
        api_key=api_key,
        base_url="https://api.deepseek.com",
    )
    response = client.chat.completions.create(
        model="deepseek-chat",
        messages=[{"role": "user", "content": "Say 'DeepSeek is connected!' and nothing else."}],
        max_tokens=20,
    )
    print(response.choices[0].message.content)

## 1.6 Snapshot — What We Have

| Component | Status |
|---|---|
| PyTorch + Transformers | Installed |
| SmolLM 135M | Loaded & generating text |
| DeepSeek API | Connected (if key set) |

This is our foundation. In Chapter 2 we'll fine-tune the model on our first task: text classification.